In [1]:
import re

import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5641 entries, 0 to 5640
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date_start  5641 non-null   object
 1   date_end    602 non-null    object
 2   event       5641 non-null   object
dtypes: object(3)
memory usage: 132.3+ KB


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

ru_stopwords = stopwords.words("russian")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


texts = df["event"].dropna().astype(str).apply(clean_text)

vectorizer = TfidfVectorizer(
    max_df=0.9,
    min_df=5,
    stop_words=ru_stopwords,
    ngram_range=(1, 3),
    sublinear_tf=True,
)
X = vectorizer.fit_transform(texts)

n_topics = 10
model = NMF(n_components=n_topics, random_state=42)
W = model.fit_transform(X)
H = model.components_

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic in enumerate(H):
    top_words = [feature_names[j] for j in topic.argsort()[:-11:-1]]
    topics[f"Topic {i + 1}"] = top_words

print(len(vectorizer.vocabulary_))
topics

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ruslan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


3313


{'Topic 1': ['парламентские выборы',
  'парламентские',
  'выборы',
  'досрочные парламентские',
  'досрочные парламентские выборы',
  'досрочные',
  'партия',
  'большинство',
  'сирии',
  'мест'],
 'Topic 2': ['человек',
  'погибли',
  'погибли человек',
  'результате',
  'человека',
  'человек погибли',
  'получили',
  'ранения',
  'ранены',
  'погибло'],
 'Topic 3': ['должность',
  'вступил',
  'должность президента',
  'президента',
  'вступил должность',
  'вступил должность президента',
  'года',
  'президент',
  'должность президент',
  'вступил должность президент'],
 'Topic 4': ['тур',
  'выборов',
  'президентских',
  'президентских выборов',
  'второй',
  'второй тур',
  'тур президентских',
  'тур президентских выборов',
  'второй тур президентских',
  'одержал'],
 'Topic 5': ['мира',
  'чемпионат',
  'чемпионат мира',
  'россия',
  'мира хоккею',
  'хоккею',
  'чемпионат мира хоккею',
  'сборная',
  'шайбой',
  'хоккею шайбой'],
 'Topic 6': ['премьер',
  'министром',
  'п

In [5]:
from sklearn.decomposition import TruncatedSVD

n_topics = 10

lsa = TruncatedSVD(
    n_components=n_topics,
    random_state=42
)

X_lsa = lsa.fit_transform(X)
feature_names = vectorizer.get_feature_names_out()

topics = {}

for i, comp in enumerate(lsa.components_):
    indices = np.argsort(np.abs(comp))[-12:]
    top_words = [feature_names[j] for j in indices]
    topics[f"Topic {i + 1}"] = top_words

topics

{'Topic 1': ['тур',
  'победу одержала',
  'победу одержал',
  'одержал',
  'одержала',
  'партия',
  'президентские выборы',
  'президентские',
  'победу',
  'парламентские выборы',
  'парламентские',
  'выборы'],
 'Topic 2': ['президента',
  'погибло',
  'получили ранения',
  'ранены',
  'ранения',
  'получили',
  'человек погибли',
  'человека',
  'результате',
  'погибли человек',
  'погибли',
  'человек'],
 'Topic 3': ['одержал',
  'погибли',
  'человек',
  'тур',
  'выборов',
  'парламентские',
  'парламентские выборы',
  'вступил должность',
  'должность президента',
  'вступил',
  'президента',
  'должность'],
 'Topic 4': ['одержал',
  'победу',
  'второй',
  'президентских выборов',
  'второй тур',
  'президентских',
  'вступил должность',
  'должность президента',
  'выборов',
  'тур',
  'вступил',
  'должность'],
 'Topic 5': ['шайбой',
  'мира хоккею шайбой',
  'хоккею шайбой',
  'одержала',
  'чемпионат мира хоккею',
  'мира хоккею',
  'хоккею',
  'сборная',
  'россия',
  '

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=30,
    doc_topic_prior=0.1,  # alpha
    topic_word_prior=0.01  # beta
)
X_lda = lda.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic_dist in enumerate(lda.components_):
    top_idx = topic_dist.argsort()[-12:][::-1]
    topics[f"Topic {i + 1}"] = [feature_names[j] for j in top_idx]

topics

{'Topic 1': ['сша',
  'союз',
  'корабля',
  'экипаж',
  'тма',
  'союз тма',
  'космического',
  'космического корабля',
  'корабля союз',
  'владимир',
  'космический',
  'путин'],
 'Topic 2': ['премьер',
  'министром',
  'премьер министром',
  'стал',
  'отставку',
  'министр',
  'премьер министр',
  'новым',
  'президент',
  'новым премьер',
  'новым премьер министром',
  'лидер'],
 'Topic 3': ['космодрома',
  'запуск',
  'открытие',
  'мире',
  'казахстан',
  'официально',
  'саммит',
  'впервые',
  'стала',
  'байконур',
  'космодрома байконур',
  'истории'],
 'Topic 4': ['россии',
  'начало',
  'территории',
  'кндр',
  'первого',
  'сша',
  'лет',
  'стран',
  'сирии',
  'решение',
  'государств',
  'власти'],
 'Topic 5': ['выборы',
  'парламентские',
  'парламентские выборы',
  'победу',
  'президентские',
  'президентские выборы',
  'партия',
  'одержал',
  'победу одержал',
  'президента',
  'тур',
  'выборов'],
 'Topic 6': ['убийство',
  'открыт',
  'нато',
  'должность пре

In [7]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

texts = df["event"]

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    min_df=5
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts)

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

2026-02-24 17:20:45,928 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-02-24 17:21:41,766 - BERTopic - Embedding - Completed ✓
2026-02-24 17:21:41,767 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-24 17:22:00,058 - BERTopic - Dimensionality - Completed ✓
2026-02-24 17:22:00,060 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-24 17:22:03,038 - BERTopic - Cluster - Completed ✓
2026-02-24 17:22:03,043 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-24 17:22:03,437 - BERTopic - Representation - Completed ✓


In [8]:
import random
from collections import defaultdict


def sample_docs_per_topic(texts, topics, n_samples=10, seed=42):
    random.seed(seed)

    topic_to_docs = defaultdict(list)
    for text, topic in zip(texts, topics):
        topic_to_docs[topic].append(text)

    for topic_id, docs in sorted(topic_to_docs.items()):
        if topic_id == -1:
            print(f"topic {topic_id} | total docs: {len(docs)}")
            continue

        print("=" * 80)
        print(f"TOPIC {topic_id} | total docs: {len(docs)}")
        print("=" * 80)

        sampled = random.sample(docs, min(n_samples, len(docs)))
        for i, doc in enumerate(sampled, 1):
            print(f"{i}. {doc}")
        print()


def save_topics_barchart(topic_model: BERTopic, out_html="topics_barchart.html", top_n_topics=30):
    """
    Сохраняет интерактивный bar chart с размерами/словами тем (BERTopic).
    """
    fig = topic_model.visualize_barchart(top_n_topics=top_n_topics, n_words=10)
    fig.write_html(out_html)
    return fig


def save_documents_scatter(topic_model, texts, topics, out_html="documents_scatter.html"):
    """
    Сохраняет интерактивный scatter документов по темам (BERTopic).
    """
    fig = topic_model.visualize_documents(docs=texts, topics=topics)
    fig.write_html(out_html)
    return fig

In [9]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1588,-1_по_выборы_победу_парламентские,"[по, выборы, победу, парламентские, парламентс...",[чемпионат мира по современному пятиборью (Тай...
1,0,131,0_протеста_против_массовые_начались,"[протеста, против, массовые, начались, беспоря...",[В Абхазии начались акции протеста оппозиции с...
2,1,107,1_союз_корабля_космического корабля_космического,"[союз, корабля, космического корабля, космичес...",[приземление космического корабля Союз ТМА-01М...
3,2,106,2_результате взрыва_взрыва_результате_погибли,"[результате взрыва, взрыва, результате, погибл...",[В результате взрыва на химическом заводе во ф...
4,3,101,3_сербии_хорватии_словакии_македонии,"[сербии, хорватии, словакии, македонии, парлам...",[Парламентские выборы в Румынии. По предварите...
5,4,99,4_самолёт_борту_на борту_все,"[самолёт, борту, на борту, все, катастрофа, са...",[Авиакатастрофа в Индонезии. Самолёт Boeing 73...
6,5,83,5_сша_президент сша_дональда_представителей,"[сша, президент сша, дональда, представителей,...",[Президент США Барак Обама подписал «Закон о д...
7,6,82,6_саммит_конференция_государств_глав,"[саммит, конференция, государств, глав, нато, ...","[саммит ШОС в Самарканде., саммит АТЭС (Манила..."
8,7,79,7_произошло_землетрясения_человек_более,"[произошло, землетрясения, человек, более, без...",[В Тбилиси произошло землетрясение магнитудой ...
9,8,73,8_сирии_войска_ирака_аль,"[сирии, войска, ирака, аль, коалиции, террорис...",[Беспилотник США в Ракке в ходе атаки ликвидир...


In [10]:
sample_docs_per_topic(texts, topics, n_samples=10)
save_topics_barchart(topic_model, out_html="topic_plots/topics_barchart_auto.html")
save_documents_scatter(topic_model, texts, topics, out_html="topic_plots/documents_scatter_auto.html")

topic -1 | total docs: 1588
TOPIC 0 | total docs: 131
1. в Калининграде состоялся крупнейший в России с 1990-х годов митинг протеста против действия властей.
2. на Пушкинской площади прошёл митинг в защиту свободы слова и телекомпании НТВ, организованный партией «Яблоко», СПС и Союзом журналистов России. В нём приняли участие 20 тыс. человек.
3. В Архангельске прошла несанкционированная властями акция протеста (марш и митинг) против строительства на станции «Шиес» полигона для мусора из Москвы;
4. Акции протеста против коррупции в высших эшелонах российской власти, прошедшие в десятках городов России.
5. В Белоруссии начались протесты против президентского указа № 222.
6. В столице Анголы Луанде разогнана антиправительственная демонстрация во главе с рэпером Иконокластой.
7. массовые беспорядки в Иране.
8. массовые беспорядки в Кондопоге (Россия).
9. антисемитские беспорядки в махачкалинском аэропорту Уйташ.
10. Расстрел демонстрации в селе Аксы, Джалал-Абадской области Кыргызстана, пр

In [11]:
# semi-supervised learning
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
    zeroshot_topic_list=[
        'природная катастрофа', 'авиакатастрофа', 'государственный переворот', 'вооруженный конфликт', 'теракт', 'протесты', 'санкции', 'спорт', 'закон'
    ],
    seed_topic_list=[
        ["землетрясение", "цунами", "извержение вулкана", "ураган", "тайфун",
         "наводнение", "оползень", "сель", "засуха", "лесной пожар"],
        ["авиакатастрофа", "крушение самолета", "пассажирский самолет",
         "на борту", "рейс", "экипаж"],
        ["государственный переворот", "госпереворот", "военный переворот",
 "свержение власти", "захват власти", "путч"],
        ["вооруженный конфликт", "война", "военные действия", "наступление", "обстрел", "ВС РФ", "ВСУ"],
        ["теракт", "смертник", "террористический акт"],
        ["санкции", "пакет санкций"],
        ["акция протеста", "протест", "массовые протесты", "беспорядки"],
        ["запуск ракеты", "космос", "спутник", "космический аппарат", "орбита", "космодром"],
        ["чемпионат мира", "спорт", "золотая медаль", "олимпийские игры", "сборная"],
        ["выборы президента", "выборы премьер-министра", "парламентские выборы"],
        ["закон", "подписание закона", "вступление в силу закона", "законопроект", "принятие закона"],
        ["Nvidia", "Microsoft", "Google", "Samsung", "Huawei", "Facebook", "Apple"]
    ]
)

topics, probs = topic_model.fit_transform(texts)

2026-02-24 17:23:06,664 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-02-24 17:24:14,084 - BERTopic - Embedding - Completed ✓
2026-02-24 17:24:14,085 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-02-24 17:24:14,284 - BERTopic - Guided - Completed ✓
2026-02-24 17:24:14,285 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-24 17:24:17,728 - BERTopic - Dimensionality - Completed ✓
2026-02-24 17:24:17,729 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-02-24 17:24:17,854 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-02-24 17:24:29,568 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-24 17:24:31,162 - BERTopic - Cluster - Completed ✓
2026-02-24 17:24:31,164 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2026-02-24 17:24:31,178 - BERTopic - Zeroshot Step 2 - Completed ✓
2026-02-24 17:24:31,179 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-24 17:24:31,502 - BERTopic - Representation - Completed ✓


In [12]:
topic_model.reduce_topics(
    texts,
    nr_topics=15
)

topics, probs = topic_model.transform(texts)
topic_model.get_topic_info()

2026-02-24 17:24:31,929 - BERTopic - Topic reduction - Reducing number of topics
2026-02-24 17:24:32,144 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-24 17:24:32,399 - BERTopic - Representation - Completed ✓
2026-02-24 17:24:32,402 - BERTopic - Topic reduction - Reduced number of topics from 92 to 15


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-02-24 17:25:36,188 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1428,-1_на_по_выборов_президента,"[на, по, выборов, президента, победу, россии, ...",[второй тур президентских выборов в Румынии. П...
1,0,1375,0_человек_результате_на_россии,"[человек, результате, на, россии, более, погиб...",[Обрушение многоэтажного жилого дома Майами. П...
2,1,894,1_победу_премьер_президента_выборов,"[победу, премьер, президента, выборов, тур, по...",[второй тур выборов президента Финляндии. Побе...
3,2,625,2_союз_человек_на_человека,"[союз, человек, на, человека, все, сша, из, ст...",[легкомоторный самолёт авиакомпании Lion Air I...
4,3,396,3_мира_по_саммит_европы,"[мира, по, саммит, европы, международный, побе...","[чемпионат мира по хоккею с шайбой., Чемпионат..."
5,4,348,4_протеста_против_массовые_президента,"[протеста, против, массовые, президента, проте...",[в России состоялись акции протеста против фал...
6,5,237,5_премьер_союза_президент_югославии,"[премьер, союза, президент, югославии, ес, евр...",[Польша стала государством-председателем Совет...
7,6,82,6_компания_выход_китая_сша,"[компания, выход, китая, сша, китае, китай, по...",[Компания «Microsoft» выпустила операционную с...
8,7,71,7_израиль_израиля_оон_сша,"[израиль, израиля, оон, сша, газа, отношения, ...",[Израиль возобновил удары по сектору Газа посл...
9,8,69,8_закон_силу_об_вступил силу,"[закон, силу, об, вступил силу, договор, подпи...",[вступил в силу принятый в Грузии Закон об ино...


In [13]:
sample_docs_per_topic(texts, topics, n_samples=10)

topic -1 | total docs: 509
TOPIC 0 | total docs: 985
1. стрельба в магазине Walmart в Эль-Пасо, штат Техас, США, в результате которой погибли 22 человека и ещё 24 получили ранения.
2. в Афганистане проведена наземная операции против укреплённого района Тора-Бора.
3. Владимир Путин высказался против создания в США национальной системы ПРО.
4. принятие совместного заявления против ядерной войны лидерами России, Китая, США, Великобритании и Франции.
5. Вышел указ президента Азербайджана Ильхама Алиева о деноминации маната.
6. в 8:32 на перегоне между станциями Автозаводская — Павелецкая в сторону центра в Московском метрополитене произошёл террористический акт, в результате которого погибли 42 человека и свыше 250 получили ранения.
7. убийство сыновей Саддама Хусейна Кусея и Удея.
8. В результате взрыва бомбы в колумбийском городе Вильявисенсио 10 человек погибли, 25 получили ранения.
9. взрыв на шахте «Листвяжная», в результате которого погиб 51 человек.
10. Стрельба в парламенте швейцар

In [14]:
save_topics_barchart(topic_model, out_html="topic_plots/topics_barchart_15_topics.html")
save_documents_scatter(topic_model, texts, topics, out_html="topic_plots/documents_scatter_15_topics.html")